In [ ]:
# ==================================================
# FULL NOTEBOOK SCRIPT – SAFE FOR 4K CONTEXT MODELS
# ==================================================

import json
import re
import os
import asyncio
from typing import List, Dict, Any, Tuple
from collections import Counter

import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ==================================================
# CONFIG
# ==================================================

RULE_PATH = "label_data/rules/label_classification.93cd0348.json"
INPUT_FILE = "label_data/VNM.xlsx"
OUTPUT_FILE = "VNM.xlsx"

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:1234/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "EMPTY")
MODEL_NAME = "llama-3.2-1b-instruct-frog-vietnamese-history"

MAX_CONCURRENCY = 12

# ---- CONTEXT SAFETY ----
MAX_PROMPT_CHARS = 3000
MAX_CONTENT_CHARS = 1200
CHUNK_OVERLAP = 200

client = OpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY
)

# ==================================================
# Utils
# ==================================================

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def chunk_text(text: str, max_len: int, overlap: int) -> List[str]:
    chunks = []
    start = 0
    length = len(text)

    while start < length:
        end = start + max_len
        chunks.append(text[start:end])
        if end >= length:
            break
        start = end - overlap

    return chunks


# ==================================================
# Load rules
# ==================================================

def load_rules(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        rules = json.load(f)
    return [r for r in rules if r.get("is_active") is True]


def build_label_lookup(rules: List[Dict[str, Any]]) -> Dict[str, Dict[str, str]]:
    return {
        r["label_id"]: {
            "label_vie": r["label_vie"],
            "label_eng": r["label_eng"]
        }
        for r in rules
    }


# ==================================================
# Rule-based classify
# ==================================================

def rule_based_classify(content: str, rules: List[Dict[str, Any]]) -> List[str]:
    content_norm = normalize_text(content)
    matched = []

    for rule in rules:
        for kw in rule.get("keywords", []):
            if normalize_text(kw) in content_norm:
                matched.append(rule["label_id"])
                break

    return matched


# ==================================================
# Prompt builder (SAFE)
# ==================================================

def build_qwen_prompt(content: str, rules: List[Dict[str, Any]]) -> str:
    label_blocks = []

    for r in rules:
        label_blocks.append(f"""
[label_id: {r["label_id"]}]
- vi: {r["label_vie"]}
- en: {r["label_eng"]}
- definition: {r["definition"]}
- include: {r.get("include_scope", "")}
- exclude: {r.get("exclude_scope", "")}
""".strip())

    labels_text = "\n\n".join(label_blocks)

    prompt = f"""
### SYSTEM INSTRUCTION
You are a strict classification engine.
Select EXACTLY ONE label_id.
Return JSON only.

### CONTENT
{content}

### LABEL SET
{labels_text}

### OUTPUT
BEGIN_OUTPUT
{{ "label_id": "..." }}
END_OUTPUT
""".strip()

    # Hard safety truncate
    if len(prompt) > MAX_PROMPT_CHARS:
        overflow = len(prompt) - MAX_PROMPT_CHARS
        content = content[:-overflow - 50]
        return build_qwen_prompt(content, rules)

    return prompt


# ==================================================
# Async LLM call
# ==================================================

async def llm_classify_async(prompt: str):
    loop = asyncio.get_running_loop()

    response = await loop.run_in_executor(
        None,
        lambda: client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Strict classification engine."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=64,
            stop=["END_OUTPUT"]
        )
    )

    raw = response.choices[0].message.content
    match = re.search(r"\{[\s\S]*?\}", raw)

    if not match:
        raise ValueError(f"Invalid LLM output: {raw}")

    return json.loads(match.group())["label_id"]


# ==================================================
# LLM classify with chunking + voting
# ==================================================

async def llm_classify_with_chunking(
    text: str,
    rules: List[Dict[str, Any]],
    label_lookup: Dict[str, Dict[str, str]],
    semaphore: asyncio.Semaphore
):

    chunks = chunk_text(text, MAX_CONTENT_CHARS, CHUNK_OVERLAP)
    votes = []

    async with semaphore:
        for chunk in chunks:
            try:
                prompt = build_qwen_prompt(chunk, rules)
                label_id = await llm_classify_async(prompt)
                if label_id in label_lookup:
                    votes.append(label_id)
            except Exception as e:
                print(f"Chunk error: {e}")

    if not votes:
        return None

    return Counter(votes).most_common(1)[0][0]


# ==================================================
# Classify single text
# ==================================================

async def classify_single_text_async(
    text: str,
    rules: List[Dict[str, Any]],
    label_lookup: Dict[str, Dict[str, str]],
    semaphore: asyncio.Semaphore
)

    unknown = {
        "label_vie": "Không xác định",
        "label_eng": "Unknown"
    }

    # Rule-based
    matched = rule_based_classify(text, rules)
    if matched and matched[0] in label_lookup:
        return text, label_lookup[matched[0]]

    # LLM fallback
    label_id = await llm_classify_with_chunking(
        text, rules, label_lookup, semaphore
    )

    if label_id and label_id in label_lookup:
        return text, label_lookup[label_id]

    return text, unknown


# ==================================================
# Async classify all texts
# ==================================================

async def async_classify_texts(
    texts: List[str],
    rules: List[Dict[str, Any]],
    label_lookup: Dict[str, Dict[str, str]],
    max_concurrency: int
) -> Dict[str, Dict[str, str]]:

    semaphore = asyncio.Semaphore(max_concurrency)

    tasks = [
        classify_single_text_async(text, rules, label_lookup, semaphore)
        for text in texts
    ]

    results = {}

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Classifying"):
        text, result = await coro
        results[text] = result

    return results


# ==================================================
# Main pipeline
# ==================================================

async def process_excel_async():
    df = pd.read_excel(INPUT_FILE)

    if "Content" not in df.columns:
        raise ValueError("Missing column: Content")

    rules = load_rules(RULE_PATH)
    label_lookup = build_label_lookup(rules)

    df["dedup_key"] = df["Content"].astype(str).apply(normalize_text)
    unique_texts = df["dedup_key"].unique().tolist()

    cache = await async_classify_texts(
        unique_texts,
        rules,
        label_lookup,
        MAX_CONCURRENCY
    )

    df["label_vie"] = df["dedup_key"].map(lambda x: cache[x]["label_vie"])
    df["label_eng"] = df["dedup_key"].map(lambda x: cache[x]["label_eng"])

    df.drop(columns=["dedup_key"], inplace=True)
    df.to_excel(OUTPUT_FILE, index=False)

    print(f"✅ Done. Output saved to {OUTPUT_FILE}")


# ==================================================
# Notebook Entrypoint
# ==================================================

def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
        if loop.is_running():
            return coro
    except RuntimeError:
        pass
    return asyncio.run(coro)


await run_async(process_excel_async())


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'